In [ ]:
import numpy as np
import pandas as pd
import os
import joblib
import json
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ================= 配置参数 =================
DATA_DIR = "./benchmark_dataset/"
OUTPUT_DIR = "./results/isolation_forest/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 模型参数
MODEL_CONFIG = {
    'n_estimators': 100,           # 树的数量
    'max_samples': 'auto',         # 每个树使用的样本数
    'contamination': 0.1,          # 异常比例（初步估计）
    'max_features': 1.0,           # 每个树使用的特征比例
    'bootstrap': False,            # 是否使用bootstrap采样
    'random_state': 42,            # 随机种子（保证可复现）
    'n_jobs': -1,                  # 使用所有CPU核心
    'verbose': 1                   # 打印训练进度
}

# 超参数搜索范围（用于验证集调优）
HPARAM_GRID = {
    'contamination': [0.01, 0.05, 0.08, 0.1, 0.12, 0.15, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'max_samples': ['auto', 0.5, 0.7, 0.9]
}

# 评估阈值
THRESHOLD_METRIC = 'f1'  # 使用F1-score选择最优阈值

In [ ]:
# ================= 数据加载函数 =================

def load_benchmark_data(data_dir):    
    """
    加载所有基准数据集
    """
    print("加载数据...")
    
    # 加载训练集（仅正常数据）
    train_df = pd.read_parquet(os.path.join(data_dir, "train.parquet"))
    train_data = np.load(os.path.join(data_dir, "train_data.npy"))
    
    # 加载验证集（正常+合成异常）
    val_df = pd.read_parquet(os.path.join(data_dir, "validation_benchmark.parquet"))
    val_data = np.load(os.path.join(data_dir, "validation_data.npy"))
    
    # 加载测试集（正常+合成异常）
    test_df = pd.read_parquet(os.path.join(data_dir, "test_benchmark.parquet"))
    test_data = np.load(os.path.join(data_dir, "test_data.npy"))
    
    print(f"训练集: {train_data.shape}, 正常={np.sum(train_df['label']==0)}, 异常={np.sum(train_df['label']==1)}")
    print(f"验证集: {val_data.shape}, 正常={np.sum(val_df['label']==0)}, 异常={np.sum(val_df['label']==1)}")
    print(f"测试集: {test_data.shape}, 正常={np.sum(test_df['label']==0)}, 异常={np.sum(test_df['label']==1)}")
    
    return {
        'train': {'data': train_data, 'df': train_df},
        'validation': {'data': val_data, 'df': val_df},
        'test': {'data': test_data, 'df': test_df}
    }

In [ ]:
# ================= 特征工程函数 =================

def extract_features(data):
    """
    从滑窗数据中提取统计特征
    Isolation Forest 无法直接处理序列数据，需要展平或提取统计特征
    """
    n_samples, window_size, n_features = data.shape
    
    # 方法1: 计算每个窗口的统计特征
    features = []
    
    for i in range(n_samples):
        window = data[i]  # (window_size, n_features)
        
        # 对每个特征计算统计量
        sample_features = []
        for j in range(n_features):
            feature_series = window[:, j]
            
            # 基本统计量
            sample_features.append(np.mean(feature_series))
            sample_features.append(np.std(feature_series))
            sample_features.append(np.min(feature_series))
            sample_features.append(np.max(feature_series))
            sample_features.append(np.median(feature_series))
            
            # 变化率统计
            if window_size > 1:
                diff = np.diff(feature_series)
                sample_features.append(np.mean(diff))
                sample_features.append(np.std(diff))
                sample_features.append(np.max(np.abs(diff)))
            else:
                sample_features.extend([0, 0, 0])
            
            # 分位数
            sample_features.append(np.percentile(feature_series, 25))
            sample_features.append(np.percentile(feature_series, 75))
            
            # 偏度和峰度
            if len(feature_series) > 2:
                sample_features.append(pd.Series(feature_series).skew())
                sample_features.append(pd.Series(feature_series).kurt())
            else:
                sample_features.extend([0, 0])
        
        features.append(sample_features)
    
    return np.array(features)

def extract_features_flat(data):
    """
    展平窗口数据（简单方法，但特征维度会很高）
    """    
    n_samples, window_size, n_features = data.shape
    return data.reshape(n_samples, window_size * n_features)

In [ ]:
# ================= 模型训练函数 =================

def train_isolation_forest(train_data, config):
    """
    训练Isolation Forest模型
    """
    print("\n训练Isolation Forest...")
    print(f"训练数据形状: {train_data.shape}")
    print(f"模型参数: {config}")
    
    model = IsolationForest(**config)
    model.fit(train_data)
    
    print("训练完成！")
    return model  

In [ ]:
# ================= 超参数调优函数 =================

def tune_hyperparameters(model, val_data, val_labels, param_grid, metric='f1'):
    """
    使用验证集进行超参数调优
    """
    print("\n超参数调优...")
    
    best_score = -1
    best_params = None
    best_threshold = None
    results = []
    
    from itertools import product
    
    # 生成所有参数组合
    keys = param_grid.keys()
    values = param_grid.values()
    
    for combination in product(*values):
        params = dict(zip(keys, combination))
        print(f"\n测试参数: {params}")
        
        # 训练模型
        model_temp = IsolationForest(**{**model.get_params(), **params, 'random_state': 42})
        model_temp.fit(val_data)
        
        # 预测
        scores = model_temp.decision_function(val_data)
        
        # 调整阈值 - 使用不同的contamination值
        contamination_vals = [0.01, 0.03, 0.05, 0.07, 0.08, 0.1, 0.12, 0.15, 0.2]
        
        for contam in contamination_vals:
            # 根据contamination设置阈值
            threshold = np.percentile(scores, contam * 100)
            pred_labels = (scores < threshold).astype(int)
            
            # 计算指标
            precision = precision_score(val_labels, pred_labels, zero_division=0)
            recall = recall_score(val_labels, pred_labels, zero_division=0)
            f1 = f1_score(val_labels, pred_labels, zero_division=0)
            
            results.append({
                'params': params,
                'contamination': contam,
                'threshold': threshold,
                'precision': precision,
                'recall': recall,
                'f1': f1
            })
            
            print(f"  contamination={contam}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")
            
            # 更新最佳参数
            if metric == 'f1' and f1 > best_score:
                best_score = f1
                best_params = params
                best_threshold = threshold
    
    # 找出最佳参数
    if best_params is None:
        # 如果没有找到更好的参数，使用默认参数
        best_params = {'contamination': 0.1, 'n_estimators': 100, 'max_samples': 'auto'}
        
        # 计算默认阈值
        scores = model.decision_function(val_data)
        best_threshold = np.percentile(scores, 10)  # 对应contamination=0.1
    
    print(f"\n最佳参数: {best_params}, F1={best_score:.4f}")
    print(f"最佳阈值: {best_threshold:.4f}")
    
    return best_params, best_threshold, results

In [94]:
# ================= 评估函数 =================

def evaluate_model(model, data, labels, threshold=None, dataset_name="Test"):
    """
    评估模型性能
    """
    print(f"\n评估 {dataset_name}...")
    
    # 获取异常分数
    scores = model.decision_function(data)
    
    # 如果提供了阈值，使用阈值进行分类  
    if threshold is not None:
        pred_labels = (scores < threshold).astype(int)
    else:
        # 否则使用模型默认预测
        pred_labels = model.predict(data)
        pred_labels = np.where(pred_labels == -1, 1, 0)  # -1为异常，1为正常
    
    # 计算指标
    precision = precision_score(labels, pred_labels, zero_division=0)
    recall = recall_score(labels, pred_labels, zero_division=0)
    f1 = f1_score(labels, pred_labels, zero_division=0)
    roc_auc = roc_auc_score(labels, scores)
    
    # 计算混淆矩阵
    cm = confusion_matrix(labels, pred_labels)
    
    # 计算ROC和PR曲线
    fpr, tpr, _ = roc_curve(labels, scores)
    precision_curve, recall_curve, _ = precision_recall_curve(labels, scores)
    
    metrics = {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'fpr': fpr,
        'tpr': tpr,
        'precision_curve': precision_curve,
        'recall_curve': recall_curve,
        'scores': scores,
        'predictions': pred_labels
    }
    
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"\n  混淆矩阵:")
    print(f"    TN={cm[0,0]}, FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}, TP={cm[1,1]}")
    wrong_idx = np.where(
    (pred_labels != labels) & (labels == 1)
    )[0]
    print(f"wrong_idx：{wrong_idx}")
    
    return metrics

In [95]:
# ================= 绘图函数 =================

def plot_roc_curve(metrics, dataset_name, save_path):
    """
    绘制ROC曲线
    """
    plt.figure(figsize=(8, 6))
    plt.plot(metrics['fpr'], metrics['tpr'], label=f'{dataset_name} (AUC={metrics["roc_auc"]:.4f})', linewidth=2)
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'ROC Curve - {dataset_name}', fontsize=14)
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_precision_recall_curve(metrics, dataset_name, save_path):
    """
    绘制Precision-Recall曲线
    """
    plt.figure(figsize=(8, 6))
    plt.plot(metrics['recall_curve'], metrics['precision_curve'], linewidth=2, 
             label=f'{dataset_name} (F1={metrics["f1"]:.4f})')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title(f'Precision-Recall Curve - {dataset_name}', fontsize=14)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_confusion_matrix(metrics, dataset_name, save_path):
    """
    绘制混淆矩阵
    """
    cm = metrics['confusion_matrix']
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Anomaly'],
                yticklabels=['Normal', 'Anomaly'])
    plt.xlabel('Predicted', fontsize=12)
    plt.ylabel('Actual', fontsize=12)
    plt.title(f'Confusion Matrix - {dataset_name}', fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_anomaly_scores_distribution(metrics, dataset_name, save_path):
    """
    绘制异常分数分布
    """
    scores = metrics['scores']
    labels = metrics.get('labels', None)
    
    plt.figure(figsize=(10, 6))
    
    if labels is not None:
        # 分别绘制正常和异常样本的分数分布
        normal_scores = scores[labels == 0]
        anomaly_scores = scores[labels == 1]
        
        plt.hist(normal_scores, bins=50, alpha=0.7, label='Normal', color='blue', density=True)
        plt.hist(anomaly_scores, bins=50, alpha=0.7, label='Anomaly', color='red', density=True)
    else:
        plt.hist(scores, bins=50, alpha=0.7, color='blue', density=True)
    
    plt.xlabel('Anomaly Score', fontsize=12)
    plt.ylabel('Density', fontsize=12)
    plt.title(f'Anomaly Scores Distribution - {dataset_name}', fontsize=14)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
  # ================= 生成报告函数 =================

def generate_report(metrics_train, metrics_val, metrics_test, config, best_params, best_threshold, output_dir):
    """
    生成评估报告
    """
    report = {
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'model_config': config,
        'best_params': best_params,
        'best_threshold': best_threshold,
        'validation_metrics': {
            'precision': float(metrics_val['precision']),
            'recall': float(metrics_val['recall']),
            'f1': float(metrics_val['f1']),
            'roc_auc': float(metrics_val['roc_auc']),
            'confusion_matrix': metrics_val['confusion_matrix'].tolist()
        },
        'test_metrics': {
            'precision': float(metrics_test['precision']),
            'recall': float(metrics_test['recall']),
            'f1': float(metrics_test['f1']),
            'roc_auc': float(metrics_test['roc_auc']),
            'confusion_matrix': metrics_test['confusion_matrix'].tolist()
        }
    }
    
    # 保存JSON报告
    with open(os.path.join(output_dir, 'evaluation_report.json'), 'w') as f:
        json.dump(report, f, indent=2)
    
    # 生成文本报告
    with open(os.path.join(output_dir, 'evaluation_report.txt'), 'w') as f:
        f.write("=" * 60 + "\n")
        f.write("ISOLATION FOREST EVALUATION REPORT\n")
        f.write("=" * 60 + "\n\n")
        
        f.write(f"Timestamp: {report['timestamp']}\n\n")
        
        f.write("Model Configuration:\n")
        f.write("-" * 40 + "\n")
        for key, value in config.items():
            f.write(f"  {key}: {value}\n")
        f.write("\n")
        
        f.write("Best Parameters (from Validation):\n")
        f.write("-" * 40 + "\n")
        for key, value in best_params.items():
            f.write(f"  {key}: {value}\n")
        f.write(f"  threshold: {best_threshold:.4f}\n\n")
        
        f.write("Validation Set Performance:\n")
        f.write("-" * 40 + "\n")
        f.write(f"  Precision: {metrics_val['precision']:.4f}\n")
        f.write(f"  Recall: {metrics_val['recall']:.4f}\n")
        f.write(f"  F1-Score: {metrics_val['f1']:.4f}\n")
        f.write(f"  ROC-AUC: {metrics_val['roc_auc']:.4f}\n")
        f.write(f"  Confusion Matrix:\n")
        cm = metrics_val['confusion_matrix']
        f.write(f"    TN={cm[0,0]}, FP={cm[0,1]}\n")
        f.write(f"    FN={cm[1,0]}, TP={cm[1,1]}\n\n")
        
        f.write("Test Set Performance:\n")
        f.write("-" * 40 + "\n")
        f.write(f"  Precision: {metrics_test['precision']:.4f}\n")
        f.write(f"  Recall: {metrics_test['recall']:.4f}\n")
        f.write(f"  F1-Score: {metrics_test['f1']:.4f}\n")
        f.write(f"  ROC-AUC: {metrics_test['roc_auc']:.4f}\n")
        f.write(f"  Confusion Matrix:\n")
        cm = metrics_test['confusion_matrix']
        f.write(f"    TN={cm[0,0]}, FP={cm[0,1]}\n")
        f.write(f"    FN={cm[1,0]}, TP={cm[1,1]}\n\n")
        
        f.write("=" * 60 + "\n")
        f.write("END OF REPORT\n")
        f.write("=" * 60 + "\n")
    
    return report

In [85]:
# ================= 主执行流程 =================
print("=" * 60)
print("ISOLATION FOREST BASELINE")
print("=" * 60)
    
# 1. 加载数据
datasets = load_benchmark_data(DATA_DIR)
    
train_data = datasets['train']['data']
train_df = datasets['train']['df']
val_data = datasets['validation']['data']
val_df = datasets['validation']['df']
test_data = datasets['test']['data']
test_df = datasets['test']['df']
    

ISOLATION FOREST BASELINE
加载数据...
训练集: (18254, 60, 27), 正常=13553, 异常=4701
验证集: (6085, 60, 27), 正常=4340, 异常=1745
测试集: (6085, 60, 27), 正常=4351, 异常=1734


In [86]:
# 2. 特征提取（从滑窗数据中提取统计特征）
print("\n特征提取...")
    
# 选择特征提取方法
use_flat = False  # 设为True使用展平方法
    
if use_flat:
    train_features = extract_features_flat(train_data)
    val_features = extract_features_flat(val_data)
    test_features = extract_features_flat(test_data)
    print(f"使用展平方法，特征维度: {train_features.shape[1]}")
else:
    train_features = extract_features(train_data)
    val_features = extract_features(val_data)
    test_features = extract_features(test_data)
    print(f"使用统计特征方法，特征维度: {train_features.shape[1]}")
    
print(f"训练特征: {train_features.shape}")
print(f"验证特征: {val_features.shape}")
print(f"测试特征: {test_features.shape}")


特征提取...
使用统计特征方法，特征维度: 324
训练特征: (18254, 324)
验证特征: (6085, 324)
测试特征: (6085, 324)


In [87]:
# 3. 初始化模型（默认参数）
model = IsolationForest(**MODEL_CONFIG)
    
# 4. 超参数调优（使用验证集）
print("\n" + "=" * 60)
print("超参数调优 (使用验证集)")
print("=" * 60)
    
val_labels = val_df['label'].values
    
# 简化版的超参数调优
best_params, best_threshold, tuning_results = tune_hyperparameters(
    model, val_features, val_labels, HPARAM_GRID, metric=THRESHOLD_METRIC
)


超参数调优 (使用验证集)

超参数调优...

测试参数: {'contamination': 0.01, 'n_estimators': 50, 'max_samples': 'auto'}
  contamination=0.01, F1=0.0277, Precision=0.4098, Recall=0.0143
  contamination=0.03, F1=0.0882, Precision=0.4645, Recall=0.0487
  contamination=0.05, F1=0.1395, Precision=0.4689, Recall=0.0819
  contamination=0.07, F1=0.1741, Precision=0.4437, Recall=0.1083
  contamination=0.08, F1=0.1810, Precision=0.4148, Recall=0.1158
  contamination=0.1, F1=0.2014, Precision=0.3892, Recall=0.1358


[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   2 out of  32 | elapsed:    0.0s remaining:    0.8s
[Parallel(n_jobs=32)]: Done  32 out of  32 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished


  contamination=0.12, F1=0.2262, Precision=0.3830, Recall=0.1605
  contamination=0.15, F1=0.2521, Precision=0.3669, Recall=0.1920
  contamination=0.2, F1=0.3093, Precision=0.3763, Recall=0.2625

测试参数: {'contamination': 0.01, 'n_estimators': 50, 'max_samples': 0.5}
  contamination=0.01, F1=0.0310, Precision=0.4590, Recall=0.0160
  contamination=0.03, F1=0.0913, Precision=0.4809, Recall=0.0504
  contamination=0.05, F1=0.1385, Precision=0.4656, Recall=0.0814
  contamination=0.07, F1=0.1750, Precision=0.4460, Recall=0.1089


[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   2 out of  32 | elapsed:    0.0s remaining:    0.8s
[Parallel(n_jobs=32)]: Done  32 out of  32 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished


  contamination=0.08, F1=0.1873, Precision=0.4292, Recall=0.1198
  contamination=0.1, F1=0.2039, Precision=0.3941, Recall=0.1375
  contamination=0.12, F1=0.2229, Precision=0.3776, Recall=0.1582
  contamination=0.15, F1=0.2543, Precision=0.3702, Recall=0.1937
  contamination=0.2, F1=0.3025, Precision=0.3681, Recall=0.2567

测试参数: {'contamination': 0.01, 'n_estimators': 50, 'max_samples': 0.7}


...

[Parallel(n_jobs=32)]: Done   2 out of  32 | elapsed:    0.1s remaining:    2.5s
[Parallel(n_jobs=32)]: Done  32 out of  32 | elapsed:    0.2s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.


  contamination=0.01, F1=0.0321, Precision=0.4754, Recall=0.0166
  contamination=0.03, F1=0.0913, Precision=0.4809, Recall=0.0504
  contamination=0.05, F1=0.1405, Precision=0.4721, Recall=0.0825
  contamination=0.07, F1=0.1686, Precision=0.4296, Recall=0.1049
  contamination=0.08, F1=0.1837, Precision=0.4209, Recall=0.1175
  contamination=0.1, F1=0.2107, Precision=0.4072, Recall=0.1421
  contamination=0.12, F1=0.2326, Precision=0.3940, Recall=0.1650
  contamination=0.15, F1=0.2596, Precision=0.3779, Recall=0.1977
  contamination=0.2, F1=0.3025, Precision=0.3681, Recall=0.2567

测试参数: {'contamination': 0.2, 'n_estimators': 200, 'max_samples': 0.9}


[Parallel(n_jobs=32)]: Done   2 out of  32 | elapsed:    0.1s remaining:    2.5s
[Parallel(n_jobs=32)]: Done  32 out of  32 | elapsed:    0.2s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s


  contamination=0.01, F1=0.0354, Precision=0.5246, Recall=0.0183
  contamination=0.03, F1=0.0892, Precision=0.4699, Recall=0.0493
  contamination=0.05, F1=0.1424, Precision=0.4787, Recall=0.0837
  contamination=0.07, F1=0.1713, Precision=0.4366, Recall=0.1066
  contamination=0.08, F1=0.1837, Precision=0.4209, Recall=0.1175
  contamination=0.1, F1=0.2082, Precision=0.4023, Recall=0.1404
  contamination=0.12, F1=0.2262, Precision=0.3830, Recall=0.1605
  contamination=0.15, F1=0.2603, Precision=0.3790, Recall=0.1983
  contamination=0.2, F1=0.3113, Precision=0.3788, Recall=0.2642

最佳参数: {'contamination': 0.01, 'n_estimators': 50, 'max_samples': 0.9}, F1=0.3120
最佳阈值: 0.1424


[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.0s finished


In [89]:
# 5. 使用最佳参数训练最终模型
print("\n" + "=" * 60)
print("训练最终模型")
print("=" * 60)
    
final_config = {**MODEL_CONFIG, **best_params}
final_model = train_isolation_forest(train_features, final_config)
    
# 保存模型
model_path = os.path.join(OUTPUT_DIR, 'isolation_forest_model.pkl')
joblib.dump(final_model, model_path)
print(f"\n模型已保存: {model_path}")
    
# 保存特征提取方法信息
feature_info = {
    'method': 'statistical' if not use_flat else 'flat',
    'n_features': train_features.shape[1],
    'window_size': train_data.shape[1],
    'n_original_features': train_data.shape[2]
}
with open(os.path.join(OUTPUT_DIR, 'feature_info.json'), 'w') as f:
    json.dump(feature_info, f, indent=2)
    


训练最终模型

训练Isolation Forest...
训练数据形状: (18254, 324)
模型参数: {'n_estimators': 50, 'max_samples': 0.9, 'contamination': 0.01, 'max_features': 1.0, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1, 'verbose': 1}


[Parallel(n_jobs=32)]: Using backend ThreadingBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done   2 out of  32 | elapsed:    0.6s remaining:    9.8s
[Parallel(n_jobs=32)]: Done  32 out of  32 | elapsed:    0.6s finished


训练完成！

模型已保存: ./results/isolation_forest/isolation_forest_model.pkl


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.2s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.2s finished


In [96]:
# 6. 评估模型
print("\n" + "=" * 60)
print("模型评估")
print("=" * 60)
    
# 验证集评估
metrics_val = evaluate_model(
    final_model, val_features, val_labels, 
    threshold=best_threshold, dataset_name="Validation"
)
metrics_val['labels'] = val_labels
    
# 测试集评估
test_labels = test_df['label'].values
metrics_test = evaluate_model(
    final_model, test_features, test_labels,
    threshold=best_threshold, dataset_name="Test"
)
metrics_test['labels'] = test_labels
    


模型评估

评估 Validation...
  Precision: 0.3395
  Recall: 0.4831
  F1-Score: 0.3988
  ROC-AUC: 0.4316

  混淆矩阵:
    TN=2700, FP=1640
    FN=902, TP=843
wrong_idx：[   0    8    9   21   24   26   30   37   38   40   43   49   59   65
   71   83   85   95   97  112  117  123  125  132  140  154  165  169
  173  183  191  212  213  216  217  218  233  234  238  242  246  247
  260  262  268  270  275  276  281  292  294  299  301  304  311  335
  346  355  362  364  374  375  382  389  397  398  400  405  419  422
  425  437  439  445  447  453  461  470  473  475  481  492  503  505
  507  516  517  518  528  532  535  536  542  553  558  581  585  606
  624  631  634  659  665  673  682  688  691  712  739  746  751  752
  753  760  771  789  791  798  799  801  816  817  827  843  845  858
  863  879  882  894  897  913  921  923  932  936  944  948  951  968
  982  987  998 1002 1003 1006 1007 1008 1009 1010 1013 1014 1016 1017
 1018 1019 1020 1022 1023 1025 1026 1029 1030 1032 1033 1045 1

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished


  Precision: 0.3300
  Recall: 0.4775
  F1-Score: 0.3903
  ROC-AUC: 0.4350

  混淆矩阵:
    TN=2670, FP=1681
    FN=906, TP=828
wrong_idx：[   0    2    3   19   24   32   36   48   57   73   84   89   93   97
  115  120  129  134  135  139  147  155  156  159  161  171  186  187
  188  192  197  202  205  211  214  231  239  240  243  257  270  284
  289  295  301  304  314  318  319  320  322  324  333  340  341  346
  354  356  357  359  363  371  380  392  412  416  417  419  421  424
  434  435  440  441  458  460  468  478  488  489  500  501  503  505
  508  513  514  516  518  519  521  527  529  538  540  541  555  556
  567  570  575  577  581  587  592  605  607  618  625  633  639  646
  659  660  667  670  674  679  681  688  691  699  714  719  723  730
  742  748  752  753  757  759  769  800  804  806  808  810  814  815
  819  824  826  843  846  855  877  889  895  897  900  908  909  927
  936  967  969  976  978  979  980  983  984  996 1001 1007 1010 1012
 1016 1023 1036

[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.0s
[Parallel(n_jobs=1)]: Done  50 out of  50 | elapsed:    0.0s finished


In [98]:
# 7. 生成可视化图表
print("\n生成可视化图表...")
    
# ROC曲线
plot_roc_curve(metrics_val, "Validation", os.path.join(OUTPUT_DIR, 'roc_curve_val.png'))
plot_roc_curve(metrics_test, "Test", os.path.join(OUTPUT_DIR, 'roc_curve_test.png'))
    
# Precision-Recall曲线
plot_precision_recall_curve(metrics_val, "Validation", os.path.join(OUTPUT_DIR, 'pr_curve_val.png'))
plot_precision_recall_curve(metrics_test, "Test", os.path.join(OUTPUT_DIR, 'pr_curve_test.png'))
    
# 混淆矩阵
plot_confusion_matrix(metrics_val, "Validation", os.path.join(OUTPUT_DIR, 'confusion_matrix_val.png'))
plot_confusion_matrix(metrics_test, "Test", os.path.join(OUTPUT_DIR, 'confusion_matrix_test.png'))
    
# 异常分数分布
plot_anomaly_scores_distribution(metrics_val, "Validation", os.path.join(OUTPUT_DIR, 'scores_distribution_val.png'))
plot_anomaly_scores_distribution(metrics_test, "Test", os.path.join(OUTPUT_DIR, 'scores_distribution_test.png'))
    


生成可视化图表...


In [99]:
# 8. 生成评估报告
print("\n生成评估报告...")
    
report = generate_report(
    None, metrics_val, metrics_test, 
    MODEL_CONFIG, best_params, best_threshold, OUTPUT_DIR
)

# 9. 打印最终结果
print("\n" + "=" * 60)
print("最终测试集结果")
print("=" * 60)
print(f"Precision: {metrics_test['precision']:.4f}")
print(f"Recall: {metrics_test['recall']:.4f}")
print(f"F1-Score: {metrics_test['f1']:.4f}")
print(f"ROC-AUC: {metrics_test['roc_auc']:.4f}")
print("=" * 60)
    
print(f"\n所有结果已保存至: {OUTPUT_DIR}")
print("生成的文件:")
print("  - isolation_forest_model.pkl (训练好的模型)")
print("  - evaluation_report.json (JSON格式报告)")
print("  - evaluation_report.txt (文本格式报告)")
print("  - roc_curve_*.png (ROC曲线)")       
print("  - pr_curve_*.png (Precision-Recall曲线)")
print("  - confusion_matrix_*.png (混淆矩阵)")
print("  - scores_distribution_*.png (异常分数分布)")
print("  - feature_info.json (特征提取信息)")


生成评估报告...

最终测试集结果
Precision: 0.3300
Recall: 0.4775
F1-Score: 0.3903
ROC-AUC: 0.4350

所有结果已保存至: ./results/isolation_forest/
生成的文件:
  - isolation_forest_model.pkl (训练好的模型)
  - evaluation_report.json (JSON格式报告)
  - evaluation_report.txt (文本格式报告)
  - roc_curve_*.png (ROC曲线)
  - pr_curve_*.png (Precision-Recall曲线)
  - confusion_matrix_*.png (混淆矩阵)
  - scores_distribution_*.png (异常分数分布)
  - feature_info.json (特征提取信息)
